In [73]:
import os, requests, json
from urllib.parse import quote
from dotenv import load_dotenv
import os, re, base64, json
from dotenv import load_dotenv
import re

from openai import OpenAI

import os, re, requests
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont, ImageOps


load_dotenv()

True

In [ ]:
def fetchRecipeByMenuName(menuName):
    '''
    메뉴명으로 식품안전나라 COOKRCP01 API에서 레시피 원본 데이터를 가져오는 함수

    입력:
        menuName: 검색할 메뉴명 문자열
                  예) "오색지라시 스시", "까르보나라 뇨끼"

    처리:
        - 메뉴명에서 공백을 제거한다.
        - .env의 FOOD_API_KEY를 사용해 API 요청을 보낸다.
        - 응답 데이터에서 첫 번째 레시피 row를 꺼낸다.
        - 레시피가 있으면 반환 완료 메시지를 출력한다.
        - 레시피가 없으면 반환 실패 메시지를 출력한다.

    출력:
        dict: 레시피 원본 데이터 1개
        None: 검색 결과가 없을 경우
    '''
    name = ''.join(menuName.split())
    url = f"http://openapi.foodsafetykorea.go.kr/api/{os.getenv('FOOD_API_KEY')}/COOKRCP01/json/1/5/RCP_NM={quote(name)}"
    data = requests.get(url).json().get("COOKRCP01", {})
    recipe = data.get("row", [None])[0]

    if recipe:
        print(f"{name} 레시피 반환 완료 !")
    else:
        print(f"{name} 레시피 반환 실패..")

    return recipe

In [25]:
fetchRecipeByMenuName("까르보나라 뇨끼")

까르보나라뇨끼 레시피 반환 완료 !


{'RCP_PARTS_DTLS': '감자 220g, 깻잎 10g, 베이컨 25g, 백일송이 25g\n다진마늘 15g, 다진양파 25g, 다진대파 25g, 흰후추 3g, 버터15g, 우유 200g\n생크림 30g, 설탕 10g, 치즈가루 15g',
 'RCP_WAY2': '끓이기',
 'MANUAL_IMG20': '',
 'MANUAL20': '',
 'RCP_SEQ': '396',
 'INFO_NA': '347.2',
 'INFO_WGT': '',
 'INFO_PRO': '13.2',
 'MANUAL_IMG13': '',
 'MANUAL_IMG14': '',
 'MANUAL_IMG15': '',
 'MANUAL_IMG16': '',
 'MANUAL_IMG10': '',
 'MANUAL_IMG11': '',
 'MANUAL_IMG12': '',
 'MANUAL_IMG17': '',
 'MANUAL_IMG18': '',
 'MANUAL_IMG19': '',
 'INFO_FAT': '20.3',
 'HASH_TAG': '',
 'MANUAL_IMG02': 'http://www.foodsafetykorea.go.kr/uploadimg/cook/20_00396_02.png',
 'MANUAL_IMG03': 'http://www.foodsafetykorea.go.kr/uploadimg/cook/20_00396_03.png',
 'RCP_PAT2': '일품',
 'MANUAL_IMG04': 'http://www.foodsafetykorea.go.kr/uploadimg/cook/20_00396_04.png',
 'MANUAL_IMG05': 'http://www.foodsafetykorea.go.kr/uploadimg/cook/20_00396_05.png',
 'MANUAL_IMG01': 'http://www.foodsafetykorea.go.kr/uploadimg/cook/20_00396_01.png',
 'MANUAL01': '1. 감자를 슬라이스 하여 찜통에 찐 후 체에

In [23]:
recipe = fetchRecipeByMenuName("까르보나라 뇨끼")

까르보나라뇨끼 레시피 반환 완료 !


In [ ]:
def parseRecipeIngredients(recipeApiData):
    '''
    레시피 API 원본 데이터에서 재료 정보를 리스트로 추출하는 함수

    입력:
        recipeApiData: fetchRecipeByMenuName 함수가 반환한 레시피 dict 데이터

    처리:
        - recipeApiData에서 RCP_PARTS_DTLS 값을 가져온다.
        - 쉼표와 줄바꿈을 기준으로 재료를 나눈다.
        - 재료명, 수량, 단위는 제거하지 않고 그대로 유지한다.
        - 불필요한 공백만 정리한다.

    출력:
        list: 재료 정보 문자열 리스트
              예) ["감자 220g", "깻잎 10g", "베이컨 25g"]
    '''
    text = recipeApiData.get("RCP_PARTS_DTLS", "")
    ingredients = [re.sub(r"\s+", " ", x).strip() for x in re.split(r",|\n", text) if x.strip()]
    print(f"재료 {len(ingredients)}개 추출 완료 !")
    return ingredients

In [20]:
ingredients = parseRecipeIngredients(recipe)
print(ingredients)

재료 13개 추출 완료 !
['감자 220g', '깻잎 10g', '베이컨 25g', '백일송이 25g', '다진마늘 15g', '다진양파 25g', '다진대파 25g', '흰후추 3g', '버터15g', '우유 200g', '생크림 30g', '설탕 10g', '치즈가루 15g']


In [ ]:
def parseRecipeSteps(recipeApiData):
    '''
    레시피 API 원본 데이터에서 조리 단계와 단계별 이미지를 추출하는 함수

    입력:
        recipeApiData: fetchRecipeByMenuName 함수가 반환한 레시피 dict 데이터

    처리:
        - MANUAL01부터 MANUAL20까지 순서대로 확인한다.
        - 비어 있지 않은 조리 설명만 추출한다.
        - 각 조리 설명과 같은 번호의 MANUAL_IMG 값을 함께 가져온다.
        - 조리 설명 안의 줄바꿈과 여러 공백은 한 칸 공백으로 정리한다.
        - 이미지가 없는 단계는 imageUrl 값을 None으로 저장한다.

    출력:
        list: 조리 단계 정보 dict 리스트
              예)
              [
                  {
                      "stepNo": 1,
                      "originalText": "1. 감자를 슬라이스 하여 찜통에 찐다.",
                      "imageUrl": "http://..."
                  },
                  {
                      "stepNo": 2,
                      "originalText": "2. 베이컨을 데쳐 송송 썬다.",
                      "imageUrl": "http://..."
                  }
              ]
    '''
    steps = []

    for i in range(1, 21):
        num = str(i).zfill(2)
        manual = recipeApiData.get(f"MANUAL{num}", "").strip()
        image = recipeApiData.get(f"MANUAL_IMG{num}", "").strip()

        if manual:
            steps.append({
                "stepNo": i,
                "originalText": re.sub(r"\s+", " ", manual),
                "imageUrl": image if image else None
            })

    print(f"조리 단계 {len(steps)}개 추출 완료 !")
    return steps

In [22]:
steps = parseRecipeSteps(recipe)

print(steps)

조리 단계 6개 추출 완료 !
[{'stepNo': 1, 'originalText': '1. 감자를 슬라이스 하여 찜통에 찐 후 체에 으깬다. 깻잎을 잘게 채 썰어서 냉수에 담가 향을 빼준다.', 'imageUrl': 'http://www.foodsafetykorea.go.kr/uploadimg/cook/20_00396_01.png'}, {'stepNo': 2, 'originalText': '2. 베이컨을 한 번 데쳐내 송송 썰어준 후 다진 마늘과 볶아준다.', 'imageUrl': 'http://www.foodsafetykorea.go.kr/uploadimg/cook/20_00396_02.png'}, {'stepNo': 3, 'originalText': '3. 으깬 감자와 깻잎을 다져서 반죽을 만들고 엄지손가락 크기로 떼어 밀가루를 뿌리고, 둥글게 빚은 후 포크로 자국을 내 뇨끼를 만든다.', 'imageUrl': 'http://www.foodsafetykorea.go.kr/uploadimg/cook/20_00396_03.png'}, {'stepNo': 4, 'originalText': '4. 만들어진 뇨끼를 삶아 건져준다.', 'imageUrl': 'http://www.foodsafetykorea.go.kr/uploadimg/cook/20_00396_04.png'}, {'stepNo': 5, 'originalText': '5. 팬에 버터를 넣고 다진양파, 다진마늘, 다진 대파, 볶은 베이컨, 새송이, 흰후추를 볶다가 우유와 생크림, 설탕을 넣고 끓여준다.', 'imageUrl': 'http://www.foodsafetykorea.go.kr/uploadimg/cook/20_00396_05.png'}, {'stepNo': 6, 'originalText': '6. 뇨끼를 넣어 한 번 더 끓여준 뒤 완성한다.', 'imageUrl': 'http://www.foodsafetykorea.go.kr/uploadimg/cook/20_00396_06.png'}]


In [ ]:
class GPTAgent:
    '''
    GPT API에 프롬프트를 보내고 JSON 형식의 응답을 받아오는 함수

    입력:
        prompt: GPT에게 전달할 사용자 프롬프트 문자열

        schema: GPT 응답이 따라야 하는 JSON Schema dict
                예)
                {
                    "type": "object",
                    "properties": {
                        "cards": {
                            "type": "array",
                            "items": { ... }
                        }
                    }
                }

    처리:
        - GPT 모델에 system_prompt와 prompt를 전달한다.
        - JSON Schema 형식에 맞춰 응답을 받는다.
        - 응답 텍스트를 json.loads로 파싱한다.

    출력:
        dict: GPT가 반환한 JSON 응답을 파이썬 dict로 변환한 값
    '''
    def __init__(self, system_prompt, model="gpt-5.4", reasoning_effort="low"):
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        self.system_prompt = system_prompt
        self.model = model
        self.reasoning_effort = reasoning_effort

    def request_json(self, prompt, schema):
        response = self.client.responses.create(
            model=self.model,
            instructions=self.system_prompt,
            input=prompt,
            reasoning={"effort": self.reasoning_effort},
            text={
                "format": {
                    "type": "json_schema",
                    "name": "response_schema",
                    "strict": True,
                    "schema": schema
                }
            }
        )
        return json.loads(response.output_text)

In [ ]:
def formatStepsForBeginner(steps, menuName, ingredients):
    '''
    레시피 조리 단계 전체를 GPT API를 사용해 초보자용 단계별 카드로 변환하는 함수

    입력:
        steps: parseRecipeSteps 함수가 반환한 조리 단계 리스트
               예)
               [
                   {
                       "stepNo": 1,
                       "originalText": "1. 감자를 슬라이스 하여 찜통에 찐다.",
                       "imageUrl": "http://..."
                   },
                   {
                       "stepNo": 2,
                       "originalText": "2. 베이컨을 데쳐 송송 썬다.",
                       "imageUrl": "http://..."
                   }
               ]

        menuName: 현재 변환할 메뉴명 문자열
                  예) "까르보나라뇨끼"

        ingredients: parseRecipeIngredients 함수가 반환한 재료 리스트
                     예) ["감자 220g", "깻잎 10g", "베이컨 25g"]

    처리:
        - GPTAgent 객체를 생성한다.
        - 메뉴명, 전체 재료, 전체 조리 단계를 프롬프트에 넣는다.
        - GPT API에 한 번 요청해서 모든 조리 단계를 초보자용 카드로 변환한다.
        - 각 단계는 title, description, tip, caution, imageUrl 형식으로 정리된다.
        - 원본 stepNo 순서와 imageUrl 값은 유지한다.
        - 변환이 끝나면 생성된 카드 개수를 출력한다.

    출력:
        list: 초보자용 단계별 레시피 카드 리스트
              예)
              [
                  {
                      "stepNo": 1,
                      "title": "감자 찌기",
                      "description": "감자를 얇게 썬 뒤 찜통에 넣고 부드러워질 때까지 쪄주세요.",
                      "tip": "감자가 젓가락으로 쉽게 들어가면 잘 익은 상태입니다.",
                      "caution": "찜통에서 꺼낼 때 뜨거우니 조심하세요.",
                      "imageUrl": "http://..."
                  }
              ]
    '''
    agent = GPTAgent(
        system_prompt="너는 요리 초보자를 위한 한식 레시피 설명을 만드는 도우미야. 원본 레시피 단계를 사용자가 따라 하기 쉬운 단계별 카드로 바꿔줘.",
        reasoning_effort="low"
    )

    schema = {
        "type": "object",
        "properties": {
            "cards": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "stepNo": {"type": "integer"},
                        "title": {"type": "string"},
                        "description": {"type": "string"},
                        "tip": {"type": "string"},
                        "caution": {"type": "string"},
                        "imageUrl": {"type": ["string", "null"]}
                    },
                    "required": ["stepNo", "title", "description", "tip", "caution", "imageUrl"],
                    "additionalProperties": False
                }
            }
        },
        "required": ["cards"],
        "additionalProperties": False
    }

    prompt = f"""
메뉴명: {menuName}
전체 재료: {ingredients}
조리 단계 목록: {steps}

위 조리 단계 전체를 요리 초보자용 단계별 카드로 변환해줘.

조건:
- 원본 stepNo 순서를 유지해줘.
- title은 10자 이내의 짧은 행동 제목으로 작성해줘.
- description은 초보자가 실제로 따라 할 수 있게 쉽게 풀어서 작성해줘.
- tip은 해당 단계에 도움이 되는 조리 팁을 작성해줘.
- caution은 주의사항을 작성하고, 없으면 빈 문자열로 작성해줘.
- imageUrl은 입력받은 값을 그대로 유지해줘.
"""

    result = agent.request_json(prompt, schema)
    print(f"초보자용 조리 카드 {len(result['cards'])}개 변환 완료 !")
    return result["cards"]

In [36]:
cards = formatStepsForBeginner(steps, recipe["RCP_NM"], ingredients)

print(cards)

초보자용 조리 카드 6개 변환 완료 !
[{'stepNo': 1, 'title': '감자 찌기', 'description': '감자는 껍질을 벗긴 뒤 얇게 슬라이스하세요. 찜통에 넣고 젓가락이 부드럽게 들어갈 때까지 찐 다음, 뜨거울 때 체에 눌러 곱게 으깨주세요. 깻잎은 잘게 채 썰어 찬물에 잠시 담가 향을 부드럽게 빼줍니다.', 'tip': '감자는 뜨거울 때 으깨야 덩어리 없이 부드럽게 잘 으깨져요.', 'caution': '찜통에서 감자를 꺼낼 때 김이 매우 뜨거우니 화상에 주의하세요.', 'imageUrl': 'http://www.foodsafetykorea.go.kr/uploadimg/cook/20_00396_01.png'}, {'stepNo': 2, 'title': '베이컨 볶기', 'description': '베이컨은 끓는 물에 살짝 데쳐 기름기와 짠맛을 줄인 뒤, 물기를 빼고 송송 썰어주세요. 팬에 베이컨과 다진 마늘을 넣고 중약불에서 마늘 향이 올라올 때까지 볶아줍니다.', 'tip': '베이컨을 한 번 데치면 소스 맛이 너무 짜지 않고 깔끔해져요.', 'caution': '마늘은 쉽게 타니 센 불보다 중약불에서 볶아주세요.', 'imageUrl': 'http://www.foodsafetykorea.go.kr/uploadimg/cook/20_00396_02.png'}, {'stepNo': 3, 'title': '뇨끼 빚기', 'description': '으깬 감자에 물기를 뺀 깻잎을 잘게 다져 넣고 섞어 반죽을 만드세요. 반죽을 엄지손가락 크기로 떼어 밀가루를 살짝 묻힌 뒤 둥글게 빚고, 포크로 살짝 눌러 자국을 내 뇨끼 모양을 만듭니다.', 'tip': '손에 밀가루를 조금 묻히면 반죽이 덜 달라붙어 모양 잡기 쉬워요.', 'caution': '반죽을 너무 오래 치대면 식감이 질겨질 수 있으니 가볍게 섞어주세요.', 'imageUrl': 'http://www.foodsafetykorea.go.kr/uploadimg/cook/20_00396_

In [49]:

def safeFileName(name):
    return re.sub(r'[\\/:*?"<>| ]+', "_", str(name).strip())

def findKoreanFont(bold=False, fontPath=None):
    if fontPath and os.path.exists(fontPath):
        return fontPath

    paths = [
        "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf" if bold else "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothicBold.ttf" if bold else "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc" if bold else "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Bold.ttc" if bold else "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/mnt/c/Windows/Fonts/malgunbd.ttf" if bold else "/mnt/c/Windows/Fonts/malgun.ttf",
        "C:/Windows/Fonts/malgunbd.ttf" if bold else "C:/Windows/Fonts/malgun.ttf",
        "/System/Library/Fonts/AppleSDGothicNeo.ttc",
    ]

    for path in paths:
        if os.path.exists(path):
            return path

    raise FileNotFoundError("한글 폰트를 찾지 못했습니다. WSL/Ubuntu라면 `sudo apt install fonts-nanum` 실행 후 다시 시도하세요.")

def getFont(size, bold=False, fontPath=None):
    return ImageFont.truetype(findKoreanFont(bold, fontPath), size)

def wrapText(draw, text, font, maxWidth):
    lines, line = [], ""
    for char in str(text):
        test = line + char
        if draw.textlength(test, font=font) <= maxWidth:
            line = test
        else:
            if line:
                lines.append(line)
            line = char
    if line:
        lines.append(line)
    return lines

def drawWrappedText(draw, text, xy, font, fill, maxWidth, lineGap=8, maxLines=None):
    x, y = xy
    lines = wrapText(draw, text, font, maxWidth)

    if maxLines:
        lines = lines[:maxLines]
        if len(wrapText(draw, text, font, maxWidth)) > maxLines:
            lines[-1] = lines[-1].rstrip() + "..."

    for line in lines:
        draw.text((x, y), line, font=font, fill=fill)
        y += font.size + lineGap

    return y

def loadStepImage(imageUrl, size):
    if not imageUrl:
        return None

    try:
        response = requests.get(imageUrl, timeout=8)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert("RGB")
        return ImageOps.fit(img, size, method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
    except:
        return None

def buildRecipeCards(cards, menuName, outputDir="recipe_cards", fontPath=None):
    recipeDir = os.path.join(outputDir, safeFileName(menuName))
    os.makedirs(recipeDir, exist_ok=True)

    resultPaths = []

    for card in cards:
        stepNo = card.get("stepNo")
        title = card.get("title", "")
        description = card.get("description", "")
        tip = card.get("tip", "")
        caution = card.get("caution", "")
        imageUrl = card.get("imageUrl")

        img = Image.new("RGB", (720, 1080), "#FFF7EA")
        draw = ImageDraw.Draw(img)

        titleFont = getFont(38, True, fontPath)
        stepFont = getFont(21, True, fontPath)
        menuFont = getFont(20, False, fontPath)
        labelFont = getFont(21, True, fontPath)
        bodyFont = getFont(24, False, fontPath)
        smallFont = getFont(20, False, fontPath)

        draw.rounded_rectangle((32, 32, 688, 1048), radius=36, fill="#FFFFFF")

        draw.rounded_rectangle((56, 56, 168, 98), radius=20, fill="#FFB84D")
        draw.text((78, 67), f"STEP {stepNo}", font=stepFont, fill="#FFFFFF")

        draw.text((56, 122), menuName, font=menuFont, fill="#888888")
        drawWrappedText(draw, title, (56, 155), titleFont, "#222222", 608, lineGap=8, maxLines=2)

        imageBox = (56, 260, 664, 550)
        stepImage = loadStepImage(imageUrl, (608, 290))

        if stepImage:
            img.paste(stepImage, (56, 260))
        else:
            draw.rounded_rectangle(imageBox, radius=24, fill="#F2EFE8")
            draw.text((295, 385), "이미지 없음", font=bodyFont, fill="#999999")

        y = 585
        draw.text((56, y), "설명", font=labelFont, fill="#FF8A00")
        y = drawWrappedText(draw, description, (56, y + 34), bodyFont, "#333333", 608, lineGap=9, maxLines=5)

        y += 24
        if tip:
            draw.rounded_rectangle((56, y, 664, y + 118), radius=22, fill="#FFF1D2")
            draw.text((82, y + 20), "TIP", font=labelFont, fill="#E28A00")
            drawWrappedText(draw, tip, (82, y + 55), smallFont, "#444444", 556, lineGap=7, maxLines=2)
            y += 142

        if caution:
            draw.rounded_rectangle((56, y, 664, y + 112), radius=22, fill="#FFE9E9")
            draw.text((82, y + 18), "주의", font=labelFont, fill="#D9534F")
            drawWrappedText(draw, caution, (82, y + 52), smallFont, "#444444", 556, lineGap=7, maxLines=2)

        filePath = os.path.join(recipeDir, f"step_{str(stepNo).zfill(2)}.png")
        img.save(filePath)
        resultPaths.append(filePath)

        print(f"STEP {stepNo} 카드 PNG 생성 완료 !")

    return resultPaths

In [50]:
cardPaths = buildRecipeCards(cards, "까르보나라뇨끼")
print(cardPaths)

STEP 1 카드 PNG 생성 완료 !
STEP 2 카드 PNG 생성 완료 !
STEP 3 카드 PNG 생성 완료 !
STEP 4 카드 PNG 생성 완료 !
STEP 5 카드 PNG 생성 완료 !
STEP 6 카드 PNG 생성 완료 !
['recipe_cards/까르보나라뇨끼/step_01.png', 'recipe_cards/까르보나라뇨끼/step_02.png', 'recipe_cards/까르보나라뇨끼/step_03.png', 'recipe_cards/까르보나라뇨끼/step_04.png', 'recipe_cards/까르보나라뇨끼/step_05.png', 'recipe_cards/까르보나라뇨끼/step_06.png']


In [53]:
import os, re, requests
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont, ImageOps

def safeFileName(name):
    return re.sub(r'[\\/:*?"<>| ]+', "_", str(name).strip())

def findKoreanFont(bold=False, fontPath=None):
    if fontPath and os.path.exists(fontPath):
        return fontPath

    paths = [
        "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf" if bold else "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothicBold.ttf" if bold else "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc" if bold else "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Bold.ttc" if bold else "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/mnt/c/Windows/Fonts/malgunbd.ttf" if bold else "/mnt/c/Windows/Fonts/malgun.ttf",
        "C:/Windows/Fonts/malgunbd.ttf" if bold else "C:/Windows/Fonts/malgun.ttf",
        "/System/Library/Fonts/AppleSDGothicNeo.ttc"
    ]

    for path in paths:
        if os.path.exists(path):
            return path

    raise FileNotFoundError("한글 폰트를 찾지 못했습니다.")

def getFont(size, bold=False, fontPath=None):
    return ImageFont.truetype(findKoreanFont(bold, fontPath), size)

def wrapText(draw, text, font, maxWidth):
    lines, line = [], ""
    for char in str(text):
        test = line + char
        if draw.textlength(test, font=font) <= maxWidth:
            line = test
        else:
            if line:
                lines.append(line)
            line = char
    if line:
        lines.append(line)
    return lines

def drawWrappedText(draw, text, xy, font, fill, maxWidth, lineGap=8, maxLines=None):
    x, y = xy
    lines = wrapText(draw, text, font, maxWidth)

    if maxLines and len(lines) > maxLines:
        lines = lines[:maxLines]
        lines[-1] = lines[-1].rstrip() + "..."

    for line in lines:
        draw.text((x, y), line, font=font, fill=fill)
        y += font.size + lineGap

    return y

def loadRecipeImage(imageUrl, size):
    if not imageUrl:
        return None

    try:
        response = requests.get(imageUrl, timeout=8)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert("RGB")
        return ImageOps.fit(img, size, method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
    except:
        return None

def buildRecipeCover(recipeData, outputDir="recipe_cards", fontPath=None):
    title = recipeData.get("title", "")
    ingredients = recipeData.get("ingredients", [])
    raw = recipeData.get("raw", {})
    imageUrl = raw.get("ATT_FILE_NO_MK", "").strip() or raw.get("ATT_FILE_NO_MAIN", "").strip()

    recipeDir = os.path.join(outputDir, safeFileName(title))
    os.makedirs(recipeDir, exist_ok=True)

    img = Image.new("RGB", (720, 1080), "#FFF7EA")
    draw = ImageDraw.Draw(img)

    badgeFont = getFont(21, True, fontPath)
    titleFont = getFont(42, True, fontPath)
    subTitleFont = getFont(20, False, fontPath)
    sectionFont = getFont(24, True, fontPath)
    bodyFont = getFont(22, False, fontPath)
    chipFont = getFont(18, False, fontPath)

    draw.rounded_rectangle((32, 32, 688, 1048), radius=36, fill="#FFFFFF")

    draw.rounded_rectangle((56, 56, 170, 98), radius=20, fill="#FFB84D")
    draw.text((86, 67), "RECIPE", font=badgeFont, fill="#FFFFFF")

    draw.text((56, 126), "오늘의 추천 메뉴", font=subTitleFont, fill="#888888")
    drawWrappedText(draw, title, (56, 160), titleFont, "#222222", 608, lineGap=8, maxLines=2)

    recipeImage = loadRecipeImage(imageUrl, (608, 320))
    if recipeImage:
        img.paste(recipeImage, (56, 270))
    else:
        draw.rounded_rectangle((56, 270, 664, 590), radius=24, fill="#F2EFE8")
        draw.text((286, 415), "이미지 없음", font=bodyFont, fill="#999999")

    draw.text((56, 628), "재료", font=sectionFont, fill="#FF8A00")

    x, y = 56, 670
    lineHeight = 44
    chipPaddingX = 16
    chipH = 34
    maxX = 664

    for ingredient in ingredients:
        textW = int(draw.textlength(ingredient, font=chipFont))
        chipW = textW + chipPaddingX * 2

        if x + chipW > maxX:
            x = 56
            y += lineHeight

        draw.rounded_rectangle((x, y, x + chipW, y + chipH), radius=17, fill="#FFF1D2")
        draw.text((x + chipPaddingX, y + 7), ingredient, font=chipFont, fill="#7A5A1A")
        x += chipW + 10

    y += 70
    draw.rounded_rectangle((56, y, 664, y + 110), radius=22, fill="#F8F5EF")
    draw.text((80, y + 18), "한눈에 보기", font=sectionFont, fill="#666666")
    drawWrappedText(draw, "이 카드 다음 장부터 단계별 조리 과정을 순서대로 따라가면 됩니다.", (80, y + 54), bodyFont, "#555555", 540, lineGap=6, maxLines=2)

    filePath = os.path.join(recipeDir, "00_cover.png")
    img.save(filePath)

    print("표지 카드 PNG 생성 완료 !")
    return filePath

In [54]:
recipeData = {
    "raw": recipe,
    "title": recipe.get("RCP_NM", ""),
    "ingredients": ingredients,
    "steps": steps,
    "cards": cards
}

coverPath = buildRecipeCover(recipeData)

print(coverPath)

표지 카드 PNG 생성 완료 !
recipe_cards/까르보나라뇨끼/00_cover.png


In [61]:
def buildRecipeOverview(recipeData, outputDir="recipe_cards", fontPath=None):
    title = recipeData.get("title", "recipe")
    recipeCards = recipeData.get("cards", [])

    recipeDir = os.path.join(outputDir, safeFileName(title))
    os.makedirs(recipeDir, exist_ok=True)

    width = 720
    topH = 190
    rowH = 170
    bottomPad = 40
    height = topH + (len(recipeCards) * rowH) + bottomPad

    img = Image.new("RGB", (width, height), "#FFF7EA")
    draw = ImageDraw.Draw(img)

    badgeFont = getFont(20, True, fontPath)
    titleFont = getFont(38, True, fontPath)
    subFont = getFont(20, False, fontPath)
    stepFont = getFont(18, True, fontPath)
    stepTitleFont = getFont(25, True, fontPath)
    descFont = getFont(19, False, fontPath)

    draw.rounded_rectangle((32, 32, width - 32, height - 32), radius=36, fill="#FFFFFF")

    draw.rounded_rectangle((56, 56, 190, 96), radius=20, fill="#FFB84D")
    draw.text((82, 66), "OVERVIEW", font=badgeFont, fill="#FFFFFF")

    draw.text((56, 122), "한눈에 보는 조리 과정", font=subFont, fill="#888888")
    drawWrappedText(draw, title, (56, 152), titleFont, "#222222", 608, lineGap=8, maxLines=1)

    y = 230

    for card in sorted(recipeCards, key=lambda x: x.get("stepNo", 0)):
        stepNo = card.get("stepNo", "")
        stepTitle = card.get("title", "")
        description = card.get("description", "")
        imageUrl = card.get("generatedImageUrl") or card.get("sourceImageUrl") or card.get("imageUrl")

        draw.rounded_rectangle((56, y, 664, y + 140), radius=24, fill="#F8F5EF")

        stepImage = loadRecipeImage(imageUrl, (130, 100))
        if stepImage:
            img.paste(stepImage, (76, y + 20))
        else:
            draw.rounded_rectangle((76, y + 20, 206, y + 120), radius=16, fill="#EDE8DF")
            draw.text((103, y + 58), "NO IMG", font=stepFont, fill="#999999")

        draw.text((226, y + 22), f"STEP {stepNo}", font=stepFont, fill="#FF8A00")
        drawWrappedText(draw, stepTitle, (226, y + 48), stepTitleFont, "#222222", 400, lineGap=5, maxLines=1)
        drawWrappedText(draw, description, (226, y + 86), descFont, "#555555", 400, lineGap=5, maxLines=2)

        y += rowH

    filePath = os.path.join(recipeDir, "01_overview.png")
    img.save(filePath)

    recipeData["overviewImagePath"] = filePath

    print("한눈에 보기 PNG 생성 완료 !")
    return filePath

In [62]:
overviewPath = buildRecipeOverview(recipeData)

print("표지 카드:", coverPath)
print("한눈에 보기:", overviewPath)

한눈에 보기 PNG 생성 완료 !
표지 카드: recipe_cards/까르보나라뇨끼/00_cover.png
한눈에 보기: recipe_cards/까르보나라뇨끼/01_overview.png


In [65]:
def safeFileName(name):
    return re.sub(r'[\\/:*?"<>| ]+', "_", str(name).strip())

def generateMainImagePrompt(recipeData):
    agent = GPTAgent(
        system_prompt="너는 음식 이미지 생성을 위한 프롬프트를 작성하는 프롬프트 엔지니어야. 완성 음식 이미지를 만들기 좋은 영어 프롬프트를 작성해. 반드시 JSON만 반환해.",
        model="gpt-5.4",
        reasoning_effort="low"
    )

    schema = {
        "type": "object",
        "properties": {
            "imagePrompt": {"type": "string"}
        },
        "required": ["imagePrompt"],
        "additionalProperties": False
    }

    prompt = f"""
메뉴명: {recipeData.get("title", "")}
재료 목록: {recipeData.get("ingredients", [])}
요리 종류: {recipeData.get("raw", {}).get("RCP_PAT2", "")}
조리 방법: {recipeData.get("raw", {}).get("RCP_WAY2", "")}

위 정보를 바탕으로 gpt-image-2에 넣을 완성 음식 대표 이미지 생성용 영어 프롬프트를 만들어줘.

조건:
- 완성된 음식 한 그릇 또는 한 접시 중심
- 2D flat illustration 스타일
- 한식 레시피 앱에 어울리는 깔끔하고 따뜻한 분위기
- 먹음직스럽고 부드러운 색감
- 음식이 화면 중앙에 잘 보이게 구성
- 배경은 단순하고 밝게
- 사람, 손, 글자, 로고, UI 요소는 절대 넣지 않기
- 프롬프트 문자열만 imagePrompt에 넣기
"""

    result = agent.request_json(prompt, schema)
    recipeData["mainImagePrompt"] = result["imagePrompt"]

    print("대표 음식 이미지 프롬프트 생성 완료 !")
    return result["imagePrompt"]

In [67]:
mainImagePrompt = generateMainImagePrompt(recipeData)
print(mainImagePrompt)


대표 음식 이미지 프롬프트 생성 완료 !
A clean, appetizing 2D flat illustration of a single plated bowl of creamy carbonara gnocchi, centered in the composition. Soft potato gnocchi coated in a rich pale cream sauce made with milk, cream, butter, garlic, onion, and cheese, topped with crispy bacon pieces, sliced white mushrooms, and a gentle garnish of chopped green onion and fresh perilla leaves. Warm, cozy Korean recipe app style, bright and simple background, minimal composition, smooth shapes, soft and delicious color palette, subtle depth, neat plating, inviting and elegant. Focus on the finished dish only, one bowl, no side dishes. No people, no hands, no utensils, no text, no logo, no UI elements.


In [68]:
def generateMainFoodImage(recipeData, outputDir="recipe_cards"):
    title = recipeData.get("title", "recipe")

    if not recipeData.get("mainImagePrompt"):
        generateMainImagePrompt(recipeData)

    recipeDir = os.path.join(outputDir, safeFileName(title))
    os.makedirs(recipeDir, exist_ok=True)

    imageAgent = GPTAgent(
        system_prompt="",
        model="gpt-5.4",
        reasoning_effort="low"
    )

    response = imageAgent.client.images.generate(
        model="gpt-image-2",
        prompt=recipeData["mainImagePrompt"],
        size="1024x1024",
        quality="medium",
        output_format="png",
        n=1
    )

    filePath = os.path.join(recipeDir, "main_food.png")

    with open(filePath, "wb") as f:
        f.write(base64.b64decode(response.data[0].b64_json))

    recipeData["mainImageUrl"] = filePath

    print("대표 음식 이미지 생성 완료 !")
    return filePath

In [74]:
mainImagePath = generateMainFoodImage(recipeData)
print(mainImagePath)

대표 음식 이미지 생성 완료 !
recipe_cards/까르보나라뇨끼/main_food.png
